# Step 4 – Select well-covered topics
**Purpose:** merge OpenAlex topic ids to the PubMed Knowledge Graph (PKG 2.0) author–paper table and keep only papers in topics
with < 20 % of citations from outside PubMed.

**Inputs:** `Final_Result.sorted.tsv` (PKG 2.0 author × paper table incl. topic and AID; output of step 3),
`outside_topic_cites_new.csv` (share of outside-PubMed references per topic; step 2).
**Outputs:** `topics_and_Final_Result.csv`, `topics_and_Final_Result_filtered.parquet`, `Final_Result.selected.csv`.
All files are read from and written to `data/derived/`.


In [ ]:
import os
import pandas as pd

BASE_DIR = os.path.abspath('..')                  # root of the replication package (notebooks run from code/)
RAW_DIR = f'{BASE_DIR}/data/raw'                   # PKG / MEDLINE / OpenAlex input files (not included)
DERIVED_DIR = f'{BASE_DIR}/data/derived'           # intermediate files written by steps 1-11
os.makedirs(DERIVED_DIR, exist_ok=True)

# import selected data (topics, all papers)

In [ ]:
df = pd.read_csv(f'{DERIVED_DIR}/Final_Result.sorted.tsv', sep='\t')

In [ ]:
df.head()

In [ ]:
len(df)

import coverage csv

In [ ]:
topic_coverage = pd.read_csv(f'{DERIVED_DIR}/outside_topic_cites_new.csv')

In [ ]:
topic_coverage.head()

In [ ]:
merged = df.merge(topic_coverage, left_on = 'topics_num', right_on = 'topics_num')

In [ ]:
merged.to_csv(f'{DERIVED_DIR}/topics_and_Final_Result.csv', index = False)

# select well-represented topics

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import duckdb

In [ ]:
merged = pd.read_csv(f'{DERIVED_DIR}/topics_and_Final_Result.csv')

In [ ]:
merged.head()

how does topic size relate to share of outside citations?

In [ ]:
# count rows per topics_num and keep share_outside_pubmed_per_topic
result = (merged.groupby('topics_num', as_index=False).agg(rows=('topics_num', 'size'),share_outside_pubmed_per_topic=('share_outside_PubMed_per_topic', 'first')))

In [ ]:
result.sort_values('rows', ascending=False)

In [ ]:
plt.scatter(result['rows'], result['share_outside_pubmed_per_topic'])

In [ ]:
merged_selected = merged[merged['share_outside_PubMed_per_topic'].notna()]

In [ ]:
merged_selected.head()

Select topics with coverage > 80 % (outside-PubMed citations ≤ 20 %). DuckDB is used because the merged table (~34 GB) does not fit into memory.

In [ ]:
duckdb.sql(f'''
    copy (
        select *
        from read_csv(
            '{DERIVED_DIR}/topics_and_Final_Result.csv',
            header=true,
            delim=',',
            quote='"',
            escape='"',
            sample_size=-1
        )
        where share_outside_PubMed_per_topic <= 0.2
    )
    to '{DERIVED_DIR}/topics_and_Final_Result_filtered.parquet'
    (format parquet)
''')

In [ ]:
merged_selected = pd.read_parquet(f'{DERIVED_DIR}/topics_and_Final_Result_filtered.parquet')

In [ ]:
merged_selected.head()

In [ ]:
merged_selected.to_csv(f'{DERIVED_DIR}/Final_Result.selected.csv', index = False)

In [ ]:
import pandas as pd

In [ ]:
merged_selected = pd.read_csv(f'{DERIVED_DIR}/Final_Result.selected.csv')

In [ ]:
len(merged_selected)